# exploracao_gold_visao_360

Exploração para modelar o `gold_visao_360` (Tela 3, Camila). A Silver já está limpa: aqui respondemos **perguntas de modelagem**, não de limpeza.

- **Entrada:** `gold.dim_uc`, `gold.fato_uc_mes`, `silver.tb_ocorrencias_rede`, `silver.tb_atendimentos` e `silver.tb_atendimentos_historico`.
- **Saída:** nenhuma (só leitura). As decisões vão para o `gold_visao_360` e para o contrato Gold → API.

**Blocos**
1. Consumo comparativo: buracos de meses, histórico curto, consumos zero ou negativos e o limiar da conta alta.
2. Interrupções por UC.
3. Chamados por UC (inclui o teste do histórico: 1º evento = ABERTURA em 20,1%, igual ao acaso).
4. Tamanho por UC para o `GET /unidades/{id}`.

In [0]:
%run ../bronze-to-silver/funcoes_comuns

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")

dim_uc = spark.table(f"{CATALOGO}.gold.dim_uc")
fato = spark.table(f"{CATALOGO}.gold.fato_uc_mes")
oc = spark.table(f"{CATALOGO}.silver.tb_ocorrencias_rede")
at = spark.table(f"{CATALOGO}.silver.tb_atendimentos")
hist = spark.table(f"{CATALOGO}.silver.tb_atendimentos_historico")

# Colunas que o contrato Gold → API pressupôs: existem?
esperadas = {
    "dim_uc": (dim_uc, ["subgrupo_tarifario", "flag_trocou_medidor", "data_instalacao_medidor"]),
    "fato_uc_mes": (fato, ["valor_energia", "tributos", "tipo_leitura", "flag_leitura_estimada",
                           "flag_primeira_leitura_pos_troca", "data_leitura", "leitura_atual"]),
}
for nome, (df, cols) in esperadas.items():
    faltam = [c for c in cols if c not in df.columns]
    print(f"{nome}: {df.count():,} linhas | faltam: {faltam or 'nenhuma'}")

print("\nflags do fato:", [c for c in fato.columns if c.startswith("flag_")])

## Bloco 1: Consumo comparativo
**1a. Histórico por UC:** buracos de meses (decide `lag(12)` × self-join), UCs com menos de 13 meses (sem ano anterior) e consumos nulos, zero ou negativos.

In [0]:
mes_dt = F.to_date("mes_referencia", "yyyy-MM")

# Meses por UC: quantos tem, e quantos deveria ter entre o primeiro e o último
por_uc = (fato.groupBy("id_uc")
              .agg(F.count("*").alias("meses"), F.min(mes_dt).alias("ini"), F.max(mes_dt).alias("fim"))
              .withColumn("esperados", F.months_between("fim", "ini").cast("int") + 1)
              .withColumn("buracos", F.col("esperados") - F.col("meses")))

print("UCs com buraco de meses:", por_uc.filter("buracos > 0").count())
print("UCs com < 13 meses (nunca têm ano anterior):", por_uc.filter("meses < 13").count())
por_uc.select(F.min("meses"), F.expr("percentile(meses, 0.5)"), F.max("meses"),
              F.min("ini"), F.max("fim")).show()

# Consumo do mês: nulo (fatura sem leitura), zero e negativo
c = F.col("consumo_kwh")
fato.agg(F.sum(c.isNull().cast("int")).alias("nulo"),
         F.sum((c == 0).cast("int")).alias("zero"),
         F.sum((c < 0).cast("int")).alias("negativo"),
         F.sum(F.col("flag_consumo_negativo").cast("int")).alias("flag_consumo_negativo"),
         F.sum(F.col("flag_primeira_leitura_pos_troca").cast("int")).alias("pos_troca")).show()

**1b. Limiar da conta alta:** distribuição da variação do consumo contra o mesmo mês do ano anterior, só nos meses comparáveis (os dois meses existem e são positivos). Os meses de primeira leitura pós-troca aparecem separados.

In [0]:
atual = fato.select("id_uc", "mes_referencia", "consumo_kwh", "flag_primeira_leitura_pos_troca")
anterior = fato.select("id_uc",
    F.date_format(F.add_months(F.to_date("mes_referencia", "yyyy-MM"), 12), "yyyy-MM").alias("mes_referencia"),
    F.col("consumo_kwh").alias("consumo_kwh_ano_anterior"))

comp = (atual.join(anterior, ["id_uc", "mes_referencia"], "left")
             .withColumn("variacao_pct", F.try_divide(F.col("consumo_kwh") - F.col("consumo_kwh_ano_anterior"),
                                                       F.col("consumo_kwh_ano_anterior"))))

validos = comp.filter((F.col("consumo_kwh") > 0) & (F.col("consumo_kwh_ano_anterior") > 0))
print(f"meses: {comp.count():,} | sem ano anterior: {comp.filter(F.col('consumo_kwh_ano_anterior').isNull()).count():,}"
      f" | comparáveis: {validos.count():,}")

# Percentis da variação, sem e com os meses pós-troca
for nome, df in [("sem pós-troca", validos.filter(~F.col("flag_primeira_leitura_pos_troca"))),
                 ("só pós-troca", validos.filter("flag_primeira_leitura_pos_troca"))]:
    p = df.agg(F.expr("percentile(variacao_pct, array(0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99))")).first()[0]
    print(f"{nome}: " + " | ".join(f"p{q}={v:+.1%}" for q, v in zip([5, 25, 50, 75, 90, 95, 99], p)))

# Quantos meses cada limiar marcaria como conta alta
v = validos.filter(~F.col("flag_primeira_leitura_pos_troca"))
total = v.count()
for lim in [0.10, 0.20, 0.30, 0.50]:
    n = v.filter(F.col("variacao_pct") > lim).count()
    print(f"acima de +{lim:.0%}: {n:,} meses ({n/total:.1%})")

## Bloco 2: Interrupções por UC
Quantas ocorrências por UC (pior caso e UCs sem nenhuma), distribuição por tipo e quantas UCs têm sobreposição ou duração no teto de 48h.

In [0]:
por_uc_oc = oc.groupBy("id_uc").count()
print(f"ocorrências: {oc.count():,} | UCs com ocorrência: {por_uc_oc.count():,} de {dim_uc.count():,}")
por_uc_oc.select(F.min("count"), F.expr("percentile(count, 0.5)"), F.expr("percentile(count, 0.99)"),
                 F.max("count")).show()

oc.groupBy("tipo").count().orderBy(F.desc("count")).show(truncate=False)

oc.agg(*[F.sum(F.col(f).cast("int")).alias(f) for f in
         ["flag_longa_duracao", "flag_duracao_teto", "flag_sobreposicao"]],
       F.countDistinct(F.when(F.col("flag_sobreposicao"), F.col("id_uc"))).alias("ucs_com_sobreposicao")).show()

## Bloco 3: Chamados por UC
Chamados por UC, reclamações de conta alta (`FATURA | VALOR ALTO`), tempo de atendimento, nota não preenchida e eventos por chamado.

In [0]:
por_uc_at = at.groupBy("id_uc").count()
print(f"chamados: {at.count():,} | UCs com chamado: {por_uc_at.count():,} de {dim_uc.count():,}")
por_uc_at.select(F.min("count"), F.expr("percentile(count, 0.5)"), F.expr("percentile(count, 0.99)"),
                 F.max("count")).show()

# "Minha conta veio alta": quantas reclamações e quantas UCs reclamam mais de uma vez
valor_alto = at.filter((F.col("assunto") == "FATURA") & (F.col("subassunto") == "VALOR ALTO"))
print(f"FATURA | VALOR ALTO: {valor_alto.count():,} chamados | UCs: {valor_alto.select('id_uc').distinct().count():,}"
      f" | UCs com 2+: {valor_alto.groupBy('id_uc').count().filter('count >= 2').count():,}")

# Tempo de atendimento em horas
tempo_h = (F.unix_timestamp("data_fechamento") - F.unix_timestamp("data_abertura")) / 3600
at.select(tempo_h.alias("h")).agg(
    F.min("h"), F.expr("percentile(h, array(0.5, 0.9, 0.99))"), F.max("h")).show(truncate=False)
print("nota não preenchida:", at.filter(F.col("nota_satisfacao").isNull()).count())

# Eventos por chamado (o que vira qtd_eventos)
hist.groupBy("protocolo").count().groupBy("count").count().orderBy("count").show()

**3b. Reclamação de conta alta × consumo:** nas reclamações `FATURA | VALOR ALTO`, a fatura reclamada (mês do chamado ou o anterior) teve consumo +20% acima do ano anterior? Comparado com a taxa de todos os meses (5,4%).

In [0]:
LIMIAR = 0.20
flag_mes = validos.select("id_uc", "mes_referencia", (F.col("variacao_pct") > LIMIAR).alias("conta_alta"))

rec = valor_alto.select("protocolo", "id_uc", F.to_date(F.date_trunc("month", "data_abertura")).alias("m"))
for nome, desloc in [("mês do chamado", 0), ("mês anterior", -1)]:
    r = (rec.withColumn("mes_referencia", F.date_format(F.add_months("m", desloc), "yyyy-MM"))
            .join(flag_mes, ["id_uc", "mes_referencia"], "inner"))
    n, alta = r.count(), r.filter("conta_alta").count()
    print(f"{nome}: {n:,} reclamações comparáveis | consumo +{LIMIAR:.0%}: {alta:,} ({alta/n:.1%})")

## Conclusões (decisões para o gold_visao_360)

**uc_consumo_comparativo** (UC × mês, 2.152.336 linhas)
- Ano anterior por self-join (`mes_referencia − 12 meses`). Hoje não há buraco de meses, mas o self-join não depende disso.
- Sem ano anterior: 539.768 meses (os 12 primeiros de cada UC e 61 UCs ligadas recentemente). Ficam nulos, nunca 0.
- `variacao_pct` nula quando falta o ano anterior ou quando algum dos dois meses é ≤ 0 (1.712 meses com consumo negativo, problema enviado ao dono do silver_medicao).
- `flag_conta_alta = variacao_pct > 20%`: é o p95 da variação (+20,6%) e marca 5,4% dos meses comparáveis. +10% marcaria 19,3% (ruído); +30%, 1,4%.
- Pós-troca de medidor (6.304 meses) tem variação maior (p90 +24,3% contra +15,4%): a tela mostra a troca junto da conta alta.
- Reclamações FATURA | VALOR ALTO: só 5,2% tiveram consumo +20%, a mesma taxa de todos os meses. Quem reclama não consumiu mais: a variação da conta vem de preço (tarifa, bandeira).

**uc_interrupcoes** (1 linha por ocorrência, 180.000)
- 41.512 UCs com ocorrência (3.488 sem nenhuma); mediana 3, máximo 81.
- Colunas do contrato + `flag_duracao_teto` (30) e `flag_sobreposicao` (331, em 318 UCs).

**uc_atendimentos** (1 linha por chamado, 318.871)
- 44.940 UCs com chamado; mediana 7, máximo 20.
- Colunas do contrato + `tempo_atendimento_h` (1h a 719h, mediana 125h). Nota não preenchida (9.566) fica nula.

**Fora:** `uc_atendimentos_historico` e `qtd_eventos`. A ação é aleatória (1º evento = ABERTURA em 20,1%, igual ao acaso entre 5 ações) e a quantidade de eventos é uniforme entre 1 e 3.